In [0]:
# Importando funçoes
from pyspark.sql.functions import col, sum, round, to_date, concat, lit

In [0]:
# Lendo a tabela da camada silver

df_gold_despesas = spark.read.table("tabela_silver_despesas_pizzaria")

In [0]:
# 2. Agrupamos o custo total por dia
df_gold_despesas = (
    df_gold_despesas
    .groupBy("data")
    .agg(round(sum("valor_gasto"), 2).alias("custo_diario"))
)

# 3. Transformação da Data para o formato real e Ordenação 
df_gold_despesas = df_gold_despesas.withColumn(
    "data_real", 
    to_date(concat(col("data"), lit("/2026")), "dd/MM/yyyy")
).orderBy("data_real")

# 4. Guardamos apenas as colunas essenciais para o cruzamento final
df_gold_despesas = df_gold_despesas.select("data_real", "custo_diario")

print("--- CUSTO DIÁRIO CONSOLIDADO ---")
display(df_gold_despesas)

In [0]:
# Salvando a tabela gold para despesas em formatod delta

df_gold_despesas.write.mode("overwrite").format("delta").saveAsTable("tabela_gold_despesas_pizzaria")